In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import io


In [ ]:
# preprogrammed policies for red & green
# each takes the current state (x, y, ...) and returns a discrete action

def green_movement(x, y, *args):
    return 4  # always move right


def red_movement(x, y, *args):
    return 1  # always move up


# discrete action space, shared by every robot (preprogrammed or learned)
# action index -> (dx, dy)
ACTIONS = {
    0: (0, 0),   # stay
    1: (0, 1),   # up
    2: (0, -1),  # down
    3: (-1, 0),  # left
    4: (1, 0),   # right
}


def transition(x, y, action):
    """RL transition function T(state, action) -> next_state, shared by all robots."""
    dx, dy = ACTIONS[action]
    return x + dx, y + dy


# need to use actions that are discetized motor commands eventually
def action_to_motor_command():
    """"""

    return motor_c

def reward(agent_pos, green_pos, red_pos):
    """Takes all the positions and calculates reward agent receives at time step t,
    """
    
    



def random_policy(x, y, *args):
    return np.random.choice(list(ACTIONS.keys()))

In [ ]:
class Vehicle:
    """A robot whose next state follows the RL formalism: policy_fn(state) -> action,
    then transition(state, action) -> next_state. Preprogrammed robots just use a
    fixed policy_fn instead of a learned one."""

    def __init__(self, color, x, y, policy_fn):
        self.color = color
        self.x = x
        self.y = y
        self.policy_fn = policy_fn
        self.history = [(x, y)]  # used for plotting

    def step(self, *policy_args):
        action = self.policy_fn(self.x, self.y, *policy_args)
        self.x, self.y = transition(self.x, self.y, action)
        self.history.append((self.x, self.y))
        return self.x, self.y


green = Vehicle("green", x=0, y=0, policy_fn=green_movement)
red = Vehicle("red", x=0, y=0, policy_fn=red_movement)
agent = Vehicle("blue", x=0, y=0, policy_fn=random_policy) # random for now


In [ ]:


N_STEPS = 10

for _ in range(N_STEPS):
    green.step()
    red.step()
    agent.step(green.x, green.y, red.x, red.y)  # policy_fn can see other robots' positions

gx, gy = zip(*green.history)
rx, ry = zip(*red.history)
bx, by = zip(*agent.history)

plt.plot(gx, gy, "o-", color="green", label="green")
plt.plot(rx, ry, "o-", color="red", label="red")
plt.plot(bx, by, "o-", color="blue", label="agent")
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Vehicle trajectories")
plt.show()


In [ ]:
def render_frame(step_idx, vehicles, bounds):
    """Render a single frame: positions of all vehicles up to step_idx."""
    fig, ax = plt.subplots()
    for v in vehicles:
        xs, ys = zip(*v.history[: step_idx + 1])
        ax.plot(xs, ys, "o-", color=v.color, label=v.color)
        ax.plot(xs[-1], ys[-1], "o", color=v.color, markersize=12)  # current position
    ax.set_xlim(bounds[0], bounds[1])
    ax.set_ylim(bounds[2], bounds[3])
    ax.legend()
    ax.set_title(f"step {step_idx}")

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)
    buf.seek(0)
    return imageio.imread(buf)


def make_gif(vehicles, path="./trajectory.gif", duration=0.3, padding=1):
    """Save a gif of all vehicles' trajectories over their recorded history."""
    n_steps = len(vehicles[0].history)

    all_x = [p[0] for v in vehicles for p in v.history]
    all_y = [p[1] for v in vehicles for p in v.history]
    bounds = (min(all_x) - padding, max(all_x) + padding, min(all_y) - padding, max(all_y) + padding)

    with imageio.get_writer(path, mode="I", duration=duration) as writer:
        for step_idx in range(n_steps):
            frame = render_frame(step_idx, vehicles, bounds)
            writer.append_data(frame)

    print(f"GIF written to {path}")


make_gif([green, red, agent])
